# 02 · LightGBM

Gradient boosting con crecimiento por hojas. Árboles chicos-medianos, mucho submuestreo de columnas (hay muchas features correlacionadas) y learning rate bajo.

Usa las features generadas por `00_features.ipynb`. Validación: 5 folds estratificados, los mismos para todos los modelos; las features que dependen de la etiqueta están recalculadas dentro de cada fold.

In [1]:
import os, time, numpy as np, pandas as pd, warnings
import features as F
warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
y = F.load_y(); folds = F.cv_folds(y)
import lightgbm as lgb

## Modelo

In [2]:
PARAMS = dict(objective="binary", learning_rate=0.02, n_estimators=1200, num_leaves=63, min_child_samples=10,
              colsample_bytree=0.3, subsample=0.8, subsample_freq=1, reg_lambda=5, n_jobs=6, verbose=-1)
SEEDS = [42, 43, 44]   # se promedian varias semillas para reducir la varianza del modelo

def fit_predict(A, ya, B, return_model=False):
    models = [lgb.LGBMClassifier(**PARAMS, random_state=s).fit(A, ya) for s in SEEDS]
    p = np.mean([m.predict_proba(B)[:, 1] for m in models], axis=0)
    return (p, models) if return_model else p

## Validación cruzada

In [3]:
oof = np.zeros(len(y)); t0 = time.time()
for f, (a, b) in enumerate(folds):
    A, B = F.load_fold(f)
    oof[b] = fit_predict(A, y[a], B)
    print(f"fold {f}: accuracy {((oof[b] > .5) == y[b]).mean():.4f}  ({time.time() - t0:.0f}s)")
print(f"\nAccuracy CV: {((oof > .5) == y).mean():.4f}")
np.save(f"{F.CACHE}/oof_lightgbm.npy", oof)

fold 0: accuracy 0.9501  (65s)


fold 1: accuracy 0.9474  (135s)


fold 2: accuracy 0.9501  (205s)


fold 3: accuracy 0.9469  (274s)


fold 4: accuracy 0.9470  (338s)

Accuracy CV: 0.9483


## Modelo final (todo train) y predicción de test

In [4]:
A, B = F.load_full()
proba, model = fit_predict(A, y, B, return_model=True)
for s, m in zip(SEEDS, model):
    m.booster_.save_model(f"models/lightgbm_seed{s}.txt")
imp = pd.Series(model[0].booster_.feature_importance("gain"), index=A.columns).sort_values(ascending=False)
print((imp / imp.sum()).head(20).round(3))
np.save(f"{F.CACHE}/test_lightgbm.npy", proba)
F.save_submission(proba, "lightgbm")["condition"].value_counts(normalize=True)

txt_bert            0.218
txt_bertin          0.078
txt_xlmr            0.073
te_sx_lt            0.068
snn3_y              0.051
listing_type_id     0.051
txt_tfidf           0.049
te_sx_tusado        0.039
s_free              0.023
te_sx_iq1           0.021
s_iq1               0.020
te_sx_wnull         0.017
snn1_y              0.016
te_seller_id        0.014
knn_p100            0.012
knn_p5              0.010
knn_p20             0.010
initial_quantity    0.009
te_category_id      0.009
duration_days       0.007
dtype: float64


condition
new     0.5274
used    0.4726
Name: proportion, dtype: float64